# CP8305 Final Project
## Notebook 3 – Modeling

**Team members:** _Add names here_  
**Date:** _Add date_

### Objectives
- Load the processed dataset.
- Split data into training and test sets.
- Train and compare multiple SKLearn models.
- Tune hyper-parameters using GridSearchCV.
- Select the best model and save it for analysis.

> **Tip:** Choose the task type that matches your business problem:
> - **Classification** – predicting a category (e.g. churn: yes/no)
> - **Regression** – predicting a continuous value (e.g. sales amount)
> - **Clustering** – discovering natural groupings with no labels

## 1. Setup

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import get_feature_target_split
from src.modeling import (
    split_data,
    # --- Classification ---
    CLASSIFICATION_MODELS,
    train_classifier,
    evaluate_classifier,
    # --- Regression ---
    REGRESSION_MODELS,
    train_regressor,
    evaluate_regressor,
    # --- Clustering ---
    train_kmeans,
    find_optimal_k,
    # --- Utilities ---
    cross_validate_model,
    grid_search,
    save_model,
)
from src.visualization import (
    plot_confusion_matrix,
    plot_feature_importance,
    plot_residuals,
    plot_elbow_curve,
    plot_cluster_scatter,
    save_figure,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 2. Load Processed Data

In [ ]:
DATA_FILE = Path('..') / 'data' / 'processed' / 'processed_dataset.csv'
df = pd.read_csv(DATA_FILE)
print(f'Shape: {df.shape}')
df.head()

## 3. Feature / Target Split

In [ ]:
# Target column used in preprocessing notebook.
TARGET = 'readmitted_binary'

X, y = get_feature_target_split(df, TARGET)
print(f'Features: {X.shape[1]}  |  Target: {y.name}  |  Classes/Values: {y.nunique()}')

In [ ]:
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

---
## 4A. Classification Models
_Skip to Section 4B for regression or 4C for clustering._

In [ ]:
# Compare multiple classifiers
clf_results = {}
for name in CLASSIFICATION_MODELS:
    print(f'\n--- {name} ---')
    pipeline = train_classifier(X_train, y_train, model_name=name, scale=True)
    metrics = evaluate_classifier(pipeline, X_test, y_test)
    clf_results[name] = metrics['accuracy']

# Summary table
print('\n=== Accuracy Summary ===')
for name, acc in sorted(clf_results.items(), key=lambda x: -x[1]):
    print(f'{name:<25} {acc:.4f}')

In [ ]:
# Train the best classifier for detailed evaluation
BEST_CLF = max(clf_results, key=clf_results.get)
print(f'Best classifier: {BEST_CLF}')

best_clf_pipeline = train_classifier(X_train, y_train, model_name=BEST_CLF, scale=True)
y_pred_clf = best_clf_pipeline.predict(X_test)

fig = plot_confusion_matrix(y_test, y_pred_clf, title=f'Confusion Matrix – {BEST_CLF}')
save_figure(fig, '03_confusion_matrix.png')
plt.show()

In [ ]:
# Cross-validation on best classifier
cv_results = cross_validate_model(best_clf_pipeline, X, y, cv=5, scoring='accuracy')

In [ ]:
# Hyper-parameter tuning (example for Random Forest)
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10],
}
rf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42)),
])
gs = grid_search(rf_pipeline, param_grid, X_train, y_train, cv=5, scoring='accuracy')
best_clf_tuned = gs.best_estimator_

---
## 4B. Regression Models
_Skip this section if your task is classification or clustering._

In [ ]:
reg_results = {}
for name in REGRESSION_MODELS:
    print(f'\n--- {name} ---')
    pipeline = train_regressor(X_train, y_train, model_name=name, scale=True)
    metrics = evaluate_regressor(pipeline, X_test, y_test)
    reg_results[name] = metrics

print('\n=== RMSE Summary ===')
for name, m in sorted(reg_results.items(), key=lambda x: x[1]['rmse']):
    print(f'{name:<25} RMSE={m["rmse"]:.4f}  R²={m["r2"]:.4f}')

In [ ]:
BEST_REG = min(reg_results, key=lambda k: reg_results[k]['rmse'])
best_reg_pipeline = train_regressor(X_train, y_train, model_name=BEST_REG, scale=True)
y_pred_reg = best_reg_pipeline.predict(X_test)

fig = plot_residuals(y_test, y_pred_reg, title=f'Residual Plot – {BEST_REG}')
save_figure(fig, '03_residuals.png')
plt.show()

---
## 4C. Clustering
_Skip this section if your task is supervised learning._

In [ ]:
from sklearn.preprocessing import StandardScaler

# Scale features before clustering
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Elbow method
elbow_data = find_optimal_k(X_scaled, k_range=range(2, 11))
fig = plot_elbow_curve(elbow_data)
save_figure(fig, '03_elbow_curve.png')
plt.show()

In [ ]:
# TODO: Set k based on the elbow plot above.
OPTIMAL_K = 3

kmeans_model, cluster_labels = train_kmeans(X_scaled, n_clusters=OPTIMAL_K)

# TODO: Replace 'feature_x' and 'feature_y' with two informative feature names.
# fig = plot_cluster_scatter(X_scaled, cluster_labels, 'feature_x', 'feature_y')
# save_figure(fig, '03_cluster_scatter.png')
# plt.show()

## 5. Feature Importance

In [ ]:
# Works for tree-based classifiers/regressors. Adapt model variable as needed.
# Assumes the last step in the pipeline is named 'classifier' or 'regressor'.
try:
    inner_model = best_clf_pipeline.named_steps.get('classifier') \
                  or best_clf_pipeline.named_steps.get('regressor')
    if hasattr(inner_model, 'feature_importances_'):
        fig = plot_feature_importance(inner_model, list(X.columns), top_n=20)
        save_figure(fig, '03_feature_importance.png')
        plt.show()
except Exception as e:
    print(f'Feature importance not available: {e}')

## 6. Save Best Model

In [ ]:
# TODO: Update the variable and filename to match the task type.
save_model(best_clf_pipeline, 'best_classifier.pkl')
# save_model(best_reg_pipeline, 'best_regressor.pkl')
# save_model(kmeans_model,       'kmeans_model.pkl')

print('Modeling complete. Proceed to Notebook 4 – Analysis & Findings.')